# Smart Offer Targeting — Exploration

Quick interactive walkthrough of the **real Starbucks Rewards offer dataset** and the
leakage-free modelling table. The heavy lifting lives in the `src/` modules; this
notebook is for poking at the data.

Run the pipeline first:
```
python -m src.download_data && python -m src.data_prep
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src import config

table = pd.read_csv(config.MODEL_TABLE)
portfolio = pd.read_csv(config.DATA_PROCESSED / 'portfolio_clean.csv')
print(table.shape)
table.head()

## The 10 real offers

In [ ]:
portfolio[['offer_type','reward','difficulty','duration','n_channels','reward_per_difficulty','offer_label']]

## Target balance and click rate by offer type

In [ ]:
print('overall click (viewed) rate:', round(table.clicked.mean(), 3))
print('overall accept (completed) rate:', round(table.accepted.mean(), 3))
table.groupby('offer_type')[['clicked','accepted']].mean().round(3)

## Does past behaviour predict future engagement?
Customers with a higher prior view-rate engage more now — evidence that the
leakage-free offer-response-history features carry real signal.

In [ ]:
bucket = pd.cut(table.prior_view_rate, [-0.01, 0.0001, 0.5, 0.99, 1.01],
                labels=['0','0-0.5','0.5-1','1.0'])
table.groupby(bucket, observed=False).clicked.mean().round(3)

## Next steps
- `python -m src.eda`   → saves figures to `reports/figures/`
- `python -m src.train` → trains 5 models + baseline, writes `reports/metrics.json`
- `python -m src.recommend` → Top-K recommendations
- `streamlit run app/streamlit_app.py` → dashboard